# EDA - Làm quen DuckDB

Chạy từng cell (Shift+Enter) để làm quen.

## Bước 1: Import và tạo connection

In [ ]:
import duckdb

# Tạo database in-memory (trong RAM, không lưu file)
conn = duckdb.connect()
print('Connected!')

## Bước 2: Load file JSONL vào bảng SQL

In [ ]:
# DuckDB tự đọc file JSONL và tạo bảng
conn.execute("""
    CREATE TABLE raw_events AS
    SELECT * FROM read_json_auto('events.jsonl')
""")

# Đếm xem có bao nhiêu rows
count = conn.execute("SELECT COUNT(*) FROM raw_events").fetchone()[0]
print(f'Loaded {count} rows')

## Bước 3: Xem thử data

In [ ]:
# .show() in kết quả dạng bảng đẹp
conn.sql("SELECT * FROM raw_events LIMIT 5").show()

## Bước 4: Khám phá data - tự viết query

Thử viết các query sau:
- Đếm distinct user_id
- Đếm null user_id
- Xem event_name có bao nhiêu loại
- Xem app_version có gì lạ
- Check duplicate event_id

In [ ]:
# Ví dụ: đếm phân bố event_name
conn.sql("SELECT event_name, COUNT(*) AS cnt FROM raw_events GROUP BY 1 ORDER BY 2 DESC").show()

In [ ]:
# Thử tự viết query ở đây...


## Bước 5: Lấy kết quả dạng pandas DataFrame

In [ ]:
# .df() convert kết quả SQL thành pandas DataFrame
df = conn.execute("SELECT event_name, COUNT(*) AS cnt FROM raw_events GROUP BY 1").df()
print(type(df))
df

## Bước 6: Tạo bảng mới từ query (CREATE TABLE AS SELECT)

In [ ]:
# Tạo bảng clean từ raw (ví dụ: chỉ lấy user_id không null)
conn.execute("""
    CREATE OR REPLACE TABLE test_clean AS
    SELECT * FROM raw_events
    WHERE user_id IS NOT NULL
""")

conn.sql("SELECT COUNT(*) FROM test_clean").show()

## Bước 7: Export ra CSV

In [ ]:
# Lấy DataFrame rồi export
result = conn.execute("SELECT event_name, COUNT(*) AS cnt FROM raw_events GROUP BY 1").df()
result.to_csv('test_output.csv', index=False)
print('Saved to test_output.csv')